## Dependencies


In [ ]:
import nltk
import random
import re
from collections import Counter
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet as wn
from nltk.corpus import brown

# Ensure required NLTK resources are available
nltk.download("brown")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")

import os
from openai import OpenAI
from pathlib import Path

client = OpenAI(
    api_key = ""
)

# Define the repo root once
REPO_ROOT = Path.cwd().parents[1] # the notebook launched from repo root

[nltk_data] Downloading package brown to
[nltk_data]     /Users/kumikonakajima/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/kumikonakajima/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/kumikonakajima/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## 1. Sample nouns from top 10,000 Brown words

In [8]:
"""
Cue word construction from the Brown corpus.

Procedure:
1. Extract word frequencies from the Brown corpus.
2. Select the top 10,000 most frequent word forms.
3. Apply linguistic heuristics (WordNet synsets + POS tagging)
   to identify candidate common nouns.
4. Randomly sample 700 candidate nouns for downstream GPT-based filtering.

Note:
- No random seed was set in the original run.
- For reproducibility, a fixed seed should be used in future runs.
"""

lemmatizer = WordNetLemmatizer()

irregular_plurals = {
    "men": "man",
    "women": "woman",
    "children": "child",
    "feet": "foot",
    "teeth": "tooth",
    "mice": "mouse",
    "geese": "goose",
    "people": "person",
    "sheep": "sheep",
    "fish": "fish",
}

# Be sure to set random seed for reproducibility
# NOTE: the original code was run without a seed. The dataset is available in the repository. 
#random.seed(42)

# --------------------
# Brown corpus frequency analysis
# --------------------
all_words = [w.lower() for w in brown.words()]
word_freq = Counter(all_words)

# Top 10,000 most frequent word types
most_frequent = [w for w, _ in word_freq.most_common(10000)]

candidate_nouns = []
rejected_words = []

for word in most_frequent:
    word = re.sub(r"[^a-z]", "", word)

    if word in irregular_plurals:
        word = irregular_plurals[word]

    lemma = lemmatizer.lemmatize(word, pos="n")

    if len(lemma) <= 2:
        rejected_words.append(word)
        continue

    synsets = wn.synsets(lemma)
    synset_pos = {syn.pos() for syn in synsets}
    pos_tag = nltk.pos_tag([lemma])[0][1]

    # Retain words with noun synsets and noun POS tags
    if "n" in synset_pos and pos_tag in {"NN", "NNS"}:
        if "_" not in lemma and not lemma.isdigit():
            candidate_nouns.append(lemma)
        else:
            rejected_words.append(word)
    else:
        rejected_words.append(word)

# Remove duplicates
candidate_nouns = list(set(candidate_nouns))

# --------------------
# Sampling candidate cues
# --------------------
num_to_sample = min(700, len(candidate_nouns))
sampled_nouns = random.sample(candidate_nouns, num_to_sample)

### Descriptive statistics (candidate nouns in Brown corpus)

In [25]:
candidate_frequencies = [word_freq[w] for w in candidate_nouns]
mean_candidate_frequency = sum(candidate_frequencies) / len(candidate_frequencies)
mean_candidate_length = sum(len(w) for w in candidate_nouns) / len(candidate_nouns)

print("Candidate nouns:", len(candidate_nouns))
print("Mean frequency of candidate nouns:", mean_candidate_frequency)
print("Mean word length of candidate nouns:", mean_candidate_length)


Candidate nouns: 4256
Mean frequency of candidate nouns: 45.38228383458647
Mean word length of candidate nouns: 6.743421052631579


## 2. GPT-based filtering

In [ ]:
"""
Filter candidate nouns using a GPT-based linguistic judgment and sample final cue set.
"""
def chunk_list(lst, chunk_size):
    """Split list into smaller chunks"""
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

def analyze_words(words):
    """Analyze words and return both pure nouns and excluded words"""
    prompt = """For each word in the list below, determine whether it is a common noun. Make sure to exclude any words that are verbs or proper nouns (such as specific places or personal names). 
    For each word, respond in this format:
    word: [KEEP/EXCLUDE] 
    
    Example:
    table: KEEP
    washington: EXCLUDE
    think: EXCLUDE
    found: EXCLUDE
    
    Words to analyze:
    {}""".format('\n'.join(words))
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano-2025-04-14",
            messages=[
                {"role": "system", "content": "You are a linguistic expert. Analyze each word and clearly state if it should be kept or excluded."},
                {"role": "user", "content": prompt}
            ],
            temperature=0
        )
        return response.choices[0].message.content.strip().split('\n')
    except Exception as e:
        print(f"Error processing batch: {e}")
        return []

# Process in chunks of 50 words
chunks = chunk_list(top700sampled_nouns, 50)
kept_words = []
excluded_words = []
output_file = 'cdat_cues_filtered500.txt'

for chunk in chunks:
    results = analyze_words(chunk)
    for result in results:
        if ': KEEP' in result:
            kept_words.append(result.split(':')[0].strip())
        elif ': EXCLUDE' in result:
            excluded_words.append(result)

print(f"Initial candidate words: {len(sampled_nouns)}")
print(f"Kept words after filtering: {len(kept_words)}")
print(f"Excluded words: {len(excluded_words)}")
print("\nFirst 20 excluded words:")
for word in excluded_words[:20]:
    print(word)

# Final cue selection
# NOTE: No random seed was set in the original run.
# For reproducibility, consider setting: random.seed(<fixed_integer>)
final_sampled_nouns = random.sample(kept_words, 500)

with open(output_file, 'w') as f:
    for word in final_sampled_nouns:
        f.write(word + '\n')

### Descriptive statistics (550 cues selected for the paper)

In [9]:
cue_file = REPO_ROOT/'cdat'/'data'/'cdat_cues_filtered550.txt'

with open(cue_file, 'r') as f:
    filtered550_words = [line.strip() for line in f.readlines()[:50]]

filtered550_words_unique = list(dict.fromkeys(filtered550_words))# Frequencies of sampled nouns in Brown corpus

sampled_frequencies = [word_freq[word] for word in filtered550_words_unique]

# Mean frequency
mean_sampled_frequency = sum(sampled_frequencies) / len(sampled_frequencies)

# Mean word length
mean_sampled_length = sum(len(word) for word in filtered550_words) / len(filtered550_words)

print("Mean frequency of chosen nouns:", mean_sampled_frequency)
print("Mean word length of chosen nouns:", mean_sampled_length)

Mean frequency of chosen nouns: 28.94
Mean word length of chosen nouns: 7.2
